# Runtime & Space Complexity - Models

O-notation analysis for each model in the pipeline. Feature engineering complexity is covered separately in `runtime_analysis_features.ipynb`. Constants drop out throughout.

---

## Notation

| Symbol | Meaning | Value / source |
|--------|---------|----------------|
| $n$ | number of training samples | Jigsaw dataset, 80 % split |
| $p$ | number of features | **32** (fixed by `DenseFeatureTransformer`) |
| $T$ | iterations (gradient methods) or trees (RF) | hyperparameter |
| $d$ | max tree depth | hyperparameter (`max_depth`) |
| $h$ | CV folds in `run_grid_search` | **3** (hardcoded in `_common.py:159`) |
| $|\mathcal{G}|$ | number of hyperparameter combinations | model-specific |

---

## 1. Baseline - `baseline/baseline.py`

```python
# baseline.py, lines 24-28
model = DummyClassifier(strategy="stratified", random_state=42)
model.fit(X_train_feat, y_train)       # O(n)
y_pred  = model.predict(X_test_feat)   # O(n_test)
y_score = model.predict_proba(X_test_feat)[:, 1]  # O(n_test)
```

**Time complexity: $O(n)$**
- `fit`: one pass over $y_\text{train}$ to count class frequencies -> $O(n)$; features in `X_train_feat` are **never read**
- `predict`: samples from the stored class distribution once per test row -> $O(n_\text{test})$; again, no feature access

**Dominant operation:** class proportion counting - a single pass over $n$ labels

**Space complexity: $O(1)$** - model stores only two class probabilities regardless of $n$ or $p$

## 2. Lasso Logistic Regression - Core Model
### `lasso_log_reg/core_logistic_regression_lasso.py`

```python
# core_logistic_regression_lasso.py, lines 45-68
for iteration in range(self.max_iter):                              # T iterations

    p_hat = self._sigmoid(X @ self.coef_ + self.intercept_)        # line 46 - O(n·p)
    residual = p_hat - y                                            # line 47 - O(n)
    weighted_residual = sw * residual                               # line 48 - O(n)
    grad_coef = (X.T @ weighted_residual) / sw_sum                 # line 49 - O(n·p)

    coef_new = self.coef_ - self.learning_rate * grad_coef         # line 51 - O(p)
    coef_new = self._soft_threshold(coef_new,                      # line 52 - O(p)
                   self.alpha * self.learning_rate)

    intercept_new = self.intercept_ - self.learning_rate * (       # line 56 - O(1)
                        weighted_residual.sum() / sw_sum)

    delta = max(                                                    # lines 58-61 - O(p)
        np.max(np.abs(coef_new - self.coef_)),
        abs(intercept_new - self.intercept_),
    )
    if delta < self.tol:                                            # line 65 - O(1)
        break
```

**Line-by-line breakdown:**

| Line | Operation | Cost | Reason |
|------|-----------|------|--------|
| 46 | `X @ self.coef_` | $O(n \cdot p)$ | matrix-vector product: $n$ dot products of length $p$ |
| 47 | `p_hat - y` | $O(n)$ | element-wise subtraction over $n$ residuals |
| 48 | `sw * residual` | $O(n)$ | element-wise weight multiplication over $n$ samples |
| 49 | `X.T @ weighted_residual` | $O(n \cdot p)$ | matrix-vector product: $p$ dot products of length $n$ |
| 51 | `coef_ - lr * grad_coef` | $O(p)$ | vector subtraction over $p$ weights |
| 52 | `_soft_threshold(...)` | $O(p)$ | element-wise `sign` + `maximum` over $p$ weights - proximal step |
| 56 | intercept update | $O(1)$ | scalar weighted mean (sum already computed in line 48) |
| 58-61 | `np.max(np.abs(...))` | $O(p)$ | $\ell_\infty$ norm over $p$ weight changes for convergence check |

**Dominant operation:** the two matrix-vector products on lines 46 and 49 - each $O(n \cdot p)$, executed every iteration; all other operations ($O(n)$, $O(p)$, $O(1)$) are asymptotically dominated

**Total time: $O(T \cdot n \cdot p)$**

**Initialization (lines 35-36):**
```python
self.coef_      = np.zeros(n_features)   # O(p)
self.intercept_ = 0.0                    # O(1)
```
- $O(p)$ - completely dominated by the loop; negligible

**Space complexity:**
- $O(n \cdot p)$ for input matrix $X$ (held in memory during fit)
- $O(n)$ for intermediate vectors `p_hat`, `residual`, `weighted_residual`
- $O(p)$ for model weights `self.coef_` - **the saved model is $O(p)$** (only 32 floats)

## 3. Ridge Logistic Regression - `ridge_log_reg/ridge.py`

```python
# ridge.py, lines 19-24
pipe = make_pipeline(LogisticRegression(
    solver="lbfgs", class_weight="balanced", max_iter=1000,
))
grid = {"clf__C": [0.01, 0.1, 1.0, 10.0]}        # 4 values
run_grid_search(NAME, OUTPUT_DIR, pipe, grid, cv=3)  # 4 × 3 = 12 fits
```

sklearn's `LogisticRegression(solver="lbfgs")` uses **L-BFGS** (quasi-Newton) with L2 penalty:

- **Per iteration:** gradient computation `X.T @ residual` -> $O(n \cdot p)$; L-BFGS parameter update using $m = 10$ stored gradient/step pairs -> $O(m \cdot p)$ - $m = 10$ is a constant, so this is $O(p)$, dominated by the gradient step
- L-BFGS uses approximate curvature information -> converges in far fewer iterations than plain gradient descent; in practice $T_\text{lbfgs} \ll T_\text{GD}$
- GridSearchCV runs $4 \times 3 = 12$ fits - 12 is a constant and drops out

**Dominant operation:** gradient computation `X.T @ residual` at $O(n \cdot p)$ per iteration

**Total time: $O(T \cdot n \cdot p)$**

**Space complexity:**
- $O(p)$ for model weights
- $O(p)$ for L-BFGS buffer ($m \cdot p$, both $m$ and $p$ are constants here)

## 4. Linear SVM - `svm/svm.py`

```python
# svm.py, lines 19-25
pipe = make_pipeline(LinearSVC(
    class_weight="balanced", random_state=42, max_iter=2000,
))
grid = {"clf__C": [0.01, 0.1, 1.0, 10.0]}        # 4 values
run_grid_search(NAME, OUTPUT_DIR, pipe, grid, cv=3)  # 4 × 3 = 12 fits
```

sklearn's `LinearSVC` uses **LIBLINEAR dual coordinate descent**:

- **Per epoch:** cycles through all $p$ feature coordinates; updating one dual variable requires a dot product with that feature column over all $n$ samples -> $O(n)$ per coordinate -> $O(n \cdot p)$ per epoch
- GridSearchCV runs $4 \times 3 = 12$ fits - 12 is a constant and drops out

**Note:** `LinearSVC` has no `predict_proba` - it is **excluded from the Ensemble** (`ensemble.py` lines 23-27) and uses `decision_function` for scoring in `run_grid_search` (`_common.py` line 197)

**Dominant operation:** coordinate-wise dot product sweep -> $O(n)$ per feature, $O(n \cdot p)$ per epoch

**Total time: $O(T \cdot n \cdot p)$**

**Space complexity: $O(p)$** - only the weight vector is stored; no support vectors kept explicitly (linear kernel)

## 5. Random Forest - `random_forest/random_forest.py`

```python
# random_forest.py, lines 19-25
pipe = make_pipeline(RandomForestClassifier(
    class_weight="balanced", random_state=42, n_jobs=-1
))
grid = {
    "clf__n_estimators":      [100, 200],      # 2
    "clf__max_depth":         [10, 20, None],  # 3
    "clf__min_samples_split": [2, 5],          # 2
}  # |G| = 2×3×2 = 12 combinations
run_grid_search(NAME, OUTPUT_DIR, pipe, grid, cv=3)  # 12 × 3 = 36 fits
```

**Per tree:**
- Bootstrap sample: $O(n)$ - sample $n$ rows with replacement
- At each split node: draw $\sqrt{p}$ random features (sklearn default); scan node samples to find best Gini split -> $O(n_\text{node} \cdot \sqrt{p})$
- With balanced splits and depth $d$: summed over all levels -> $O(n \cdot \log n \cdot \sqrt{p})$ per tree (average case)
- Total for $T$ trees: $O(T \cdot n \cdot \log n \cdot \sqrt{p})$
- GridSearchCV runs $12 \times 3 = 36$ fits - 36 is a constant and drops out; trees parallelised via `n_jobs=-1`

**Dominant operation:** finding the best split at every node - scanning $\sqrt{p}$ features over node samples; the extra $\log n \cdot \sqrt{p}$ factor makes RF asymptotically more expensive per sample than the linear models

**Total time: $O(T \cdot n \cdot \log n \cdot \sqrt{p})$**

**Space complexity: $O(T \cdot 2^d)$** - all decision nodes across all $T$ trees; grows exponentially in depth and linearly in number of trees

## 6. Ensemble (Soft Voting) - `ensemble/ensemble.py`

```python
# ensemble.py, lines 50-57
voter = VotingClassifier(estimators=members, voting="soft")
voter.estimators_       = [m for _, m in members]   # pre-fitted - no training
voter.named_estimators_ = dict(members)
voter.le_               = LabelEncoder().fit([0, 1])
voter.classes_          = voter.le_.classes_

y_pred  = voter.predict(X_test_feat)               # O(n·p) per member × 3 members
y_score = voter.predict_proba(X_test_feat)[:, 1]   # average over 3 probability vectors: O(n)
```

**Training: $O(1)$** - no fitting at all; the three member pipelines (Lasso, Ridge, RF) are loaded from `.pkl` files on disk (`lines 37-42`) and `VotingClassifier` attributes are set directly by bypassing `fit()`

**Prediction:**
- Each of the 3 members runs inference -> $O(n \cdot p)$ each -> $O(3 \cdot n \cdot p) = O(n \cdot p)$
- Soft vote: element-wise average of 3 probability vectors -> $O(n)$

**Note:** SVM is excluded - `LinearSVC` has no `predict_proba`; stated explicitly in `ensemble.py` line 23

**Dominant operation:** inference through the three member models; Lasso and Ridge at $O(n \cdot p)$, RF at $O(n \cdot T \cdot d)$ (tree traversal per sample)

**Total time: $O(n \cdot p)$** (inference only, no training)

**Space complexity: $O(1)$ extra** - all memory is already occupied by the loaded member models

## 7. Inference Complexity (`predict` / `predict_proba`)

Inference runs on $n_\text{test}$ samples after training is complete.

| Model | Code | Inference time | Reason |
|-------|------|---------------|--------|
| Baseline | `baseline.py:27` | $O(n_\text{test})$ | samples from stored 2-class distribution; no feature access |
| **Lasso Log Reg** | `core_lasso.py:85` | $O(n_\text{test} \cdot p)$ | one matrix-vector product `sigmoid(X @ coef_ + intercept_)` |
| Ridge Log Reg | sklearn internals | $O(n_\text{test} \cdot p)$ | same structure: `X @ coef_` internally |
| LinearSVM | `_common.py:197` | $O(n_\text{test} \cdot p)$ | `decision_function = X @ w + b`, no sigmoid needed |
| Random Forest | sklearn internals | $O(n_\text{test} \cdot T \cdot d)$ | traverse $T$ trees each of depth $d$ per sample |
| Ensemble | `ensemble.py:56-57` | $O(n_\text{test} \cdot p)$ | calls all 3 members; RF term $O(T \cdot d)$ dominated by $O(p)$ for fixed $T, d$ |

- All linear models (Lasso, Ridge, SVM) have identical inference complexity $O(n_\text{test} \cdot p)$ - one dot product per sample
- RF inference is $O(T \cdot d)$ per sample - no matrix multiplication, just tree traversal; with `max_depth=10` and $T=200$ this constant is larger than $p=32$
- Baseline is the only model that never touches the feature matrix during inference

---

## Summary

| Model | Code location | Dominant operation | Time | Space (model) |
|-------|--------------|-------------------|------|---------------|
| Baseline | `baseline.py:25` | class proportion count | $O(n)$ | $O(1)$ |
| **Lasso Log Reg (ours)** | `core_lasso.py:46,49` | `X @ coef_` per iteration | $O(T \cdot n \cdot p)$ | $O(p)$ |
| Ridge Log Reg | `ridge.py:19-24` | L-BFGS gradient per iteration | $O(T \cdot n \cdot p)$ | $O(p)$ |
| LinearSVM | `svm.py:19-25` | coord. descent per epoch | $O(T \cdot n \cdot p)$ | $O(p)$ |
| Random Forest | `random_forest.py:19-25` | Gini split search per node | $O(T \cdot n \cdot \log n \cdot \sqrt{p})$ | $O(T \cdot 2^d)$ |
| Ensemble | `ensemble.py:50-57` | inference over 3 members | $O(n \cdot p)$ | - |

GridSearch multipliers (12× for Ridge/SVM, 36× for RF) are constants and drop out of O-notation.

- Lasso, Ridge and LinearSVM all land in $O(T \cdot n \cdot p)$ - same asymptotic class; the difference is the constant inside: L-BFGS (Ridge) needs fewer iterations than proximal gradient (Lasso) or dual coordinate descent (SVM)
- RF carries an extra $\log n \cdot \sqrt{p}$ factor and grows quadratically in memory with depth $d$; Lasso stores only $p = 32$ weights
- Ensemble training is $O(1)$ since models are loaded from disk; inference is $O(n \cdot p)$